# PathSnap preprocessing

Steps: load data, project to metres, label GPS points from the route, generate candidate arcs, build features, save, sanity-check.

**Step 1: load the data and look at one trip.**

In [1]:
import os, time
import numpy as np
import pandas as pd

ROOT = os.path.join('data', 'Student folder')   # train/ (80 trips) and student_test/ (10 trips)

def load_trip(part, tid):
    """Return track (lon, lat, t), nodes (lon, lat), arcs (from, to), route (arc ids)."""
    d = os.path.join(ROOT, part, tid)
    track = np.loadtxt(os.path.join(d, tid + '.track'), ndmin=2)
    nodes = np.loadtxt(os.path.join(d, tid + '.nodes'), ndmin=2)
    arcs  = np.loadtxt(os.path.join(d, tid + '.arcs'), dtype=int, ndmin=2)
    route = np.loadtxt(os.path.join(d, tid + '.route'), dtype=int, ndmin=1)
    return track, nodes, arcs, route

train_ids = sorted(os.listdir(os.path.join(ROOT, 'train')))
test_ids  = sorted(os.listdir(os.path.join(ROOT, 'student_test')))
print(len(train_ids), 'train trips,', len(test_ids), 'student-test trips')

80 train trips, 10 student-test trips


In [2]:
tid = test_ids[0]
track, nodes, arcs, route = load_trip('student_test', tid)
print('trip', tid)
print('track', track.shape, '-> lon, lat, time(s)')
print('nodes', nodes.shape, '-> lon, lat (row index = node id)')
print('arcs ', arcs.shape,  '-> from_node, to_node (row index = arc id)')
print('route', route.shape, '-> true arc ids in travel order')

trip 00000000
track (2503, 3) -> lon, lat, time(s)
nodes (8542, 2) -> lon, lat (row index = node id)
arcs  (18196, 2) -> from_node, to_node (row index = arc id)
route (179,) -> true arc ids in travel order


## Step 2: project to metres and label one trip

GPS gives angles, not distances, so we convert to a local flat metre grid. The route has no per-point label, so each GPS point is assigned a route arc, moving only forward along the route (dynamic programming).

In [3]:
def to_xy(lon, lat, lon0, lat0):
    """Local flat projection to metres. Origin = first GPS point of the trip."""
    x = (lon - lon0) * 111320.0 * np.cos(np.radians(lat0))
    y = (lat - lat0) * 110540.0
    return x, y

def project_trip(track, nodes):
    lon0, lat0 = track[0, 0], track[0, 1]
    gx, gy = to_xy(track[:, 0], track[:, 1], lon0, lat0)
    nx_, ny_ = to_xy(nodes[:, 0], nodes[:, 1], lon0, lat0)
    return gx, gy, nx_, ny_

gx, gy, nx_, ny_ = project_trip(track, nodes)
print('GPS extent (m):', round(np.ptp(gx)), 'x', round(np.ptp(gy)))

GPS extent (m): 44220 x 19235


In [4]:
def point_seg_dist(px, py, ax, ay, bx, by):
    """Distance from points (T,1) to segments (1,R). Returns (T,R)."""
    dx, dy = bx - ax, by - ay
    L = dx * dx + dy * dy
    t = np.clip(((px - ax) * dx + (py - ay) * dy) / np.where(L == 0, 1, L), 0, 1)
    return np.hypot(px - (ax + t * dx), py - (ay + t * dy))

A = arcs[route]
ax, ay = nx_[A[:, 0]][None, :], ny_[A[:, 0]][None, :]
bx, by = nx_[A[:, 1]][None, :], ny_[A[:, 1]][None, :]
D = point_seg_dist(gx[:, None], gy[:, None], ax, ay, bx, by)
print('distance matrix (GPS points x route arcs):', D.shape)

distance matrix (GPS points x route arcs): (2503, 179)


In [5]:
def monotone_labels(D):
    """One route arc per GPS point, never moving backwards along the route, minimising total distance."""
    T, R = D.shape
    cost = D[0].copy()
    bp = np.zeros((T, R), dtype=np.int32)
    ar = np.arange(R)
    for t in range(1, T):
        pm = np.minimum.accumulate(cost)
        bp[t] = np.maximum.accumulate(np.where(cost == pm, ar, 0))
        cost = D[t] + pm
    j = int(cost.argmin())
    idx = np.empty(T, dtype=int)
    for t in range(T - 1, -1, -1):
        idx[t] = j
        j = bp[t, j]
    return idx

route_pos = monotone_labels(D)
label_arc = route[route_pos]
d = D[np.arange(len(D)), route_pos]
print('points labelled:', len(label_arc))
print('route arcs used:', len(np.unique(route_pos)), 'of', len(route))
print('distance to labelled arc (m): median %.1f, p95 %.1f, max %.1f' % (np.median(d), np.percentile(d, 95), d.max()))
print('points > 30 m from their labelled arc:', int((d > 30).sum()))
seg_len = np.hypot(bx - ax, by - ay).ravel()
print('route arc length (m): median %.0f, p95 %.0f, max %.0f' % (np.median(seg_len), np.percentile(seg_len, 95), seg_len.max()))
print('total route length: %.1f km, trip duration: %.0f s' % (seg_len.sum() / 1000, track[-1, 2] - track[0, 2]))

points labelled: 2503
route arcs used: 174 of 179
distance to labelled arc (m): median 6.0, p95 16.6, max 35.9
points > 30 m from their labelled arc: 2
route arc length (m): median 122, p95 1305, max 2343
total route length: 51.2 km, trip duration: 2503 s


## Step 3: label every trip and choose the candidate radius

A point is *missed* if its true arc is farther than the radius R from it. The table shows the miss rate for several radii (train trips only).

In [6]:
def label_trip(track, nodes, arcs, route):
    """Needs the route, so only used for train / student_test."""
    gx, gy, nx_, ny_ = project_trip(track, nodes)
    A = arcs[route]
    D = point_seg_dist(gx[:, None], gy[:, None], nx_[A[:, 0]][None, :], ny_[A[:, 0]][None, :],
                       nx_[A[:, 1]][None, :], ny_[A[:, 1]][None, :])
    pos = monotone_labels(D)
    return route[pos], D[np.arange(len(D)), pos]

LABELS = {}   # (partition, trip) -> (true arc id per point, distance to it in metres)
t0 = time.time()
for part, ids in [('train', train_ids), ('student_test', test_ids)]:
    for tid in ids:
        LABELS[(part, tid)] = label_trip(*load_trip(part, tid))
print('labelled', len(LABELS), 'trips in %.0f s' % (time.time() - t0))

train_d = np.concatenate([v[1] for k, v in LABELS.items() if k[0] == 'train'])
print('train: distance to true arc median %.1f m, p95 %.1f m, p99 %.1f m, max %.1f m' % (np.median(train_d), *np.percentile(train_d, [95, 99]), train_d.max()))
for R_ in (20, 30, 40, 50, 75, 100):
    print(f'R = {R_:3d} m -> {100 * (train_d > R_).mean():.2f}% of train points would miss their true arc')

labelled 90 trips in 12 s
train: distance to true arc median 5.7 m, p95 141.2 m, p99 204.7 m, max 643.2 m
R =  20 m -> 14.43% of train points would miss their true arc
R =  30 m -> 12.50% of train points would miss their true arc
R =  40 m -> 11.65% of train points would miss their true arc
R =  50 m -> 11.16% of train points would miss their true arc
R =  75 m -> 10.26% of train points would miss their true arc
R = 100 m -> 9.85% of train points would miss their true arc


In [7]:
# Which trips cause the misses? (labels are only as good as the reference route)
rows = []
for (part, tid), (lab, dl) in LABELS.items():
    trk = load_trip(part, tid)[0]
    dt = np.diff(trk[:, 2])
    rows.append(dict(part=part, trip=tid, n_points=len(lab), pct_over_50m=100 * (dl > 50).mean(),
                     median_dist_m=np.median(dl), max_dist_m=dl.max(), max_gap_s=dt.max()))
Q = pd.DataFrame(rows).sort_values('pct_over_50m', ascending=False)
Q['suspect'] = Q['pct_over_50m'] > 5
Q.to_csv(os.path.join('results', 'trip_label_quality.csv'), index=False) if os.path.isdir('results') else None
print('suspect trips (>5% of points farther than 50 m from their labelled arc):', int(Q['suspect'].sum()), 'of', len(Q))
Q.head(10)

suspect trips (>5% of points farther than 50 m from their labelled arc): 5 of 90


,part,trip,n_points,pct_over_50m,median_dist_m,max_dist_m,max_gap_s,suspect
44,train,00000054,19313,89.426811,139.199259,239.956561,79.0,True
26,train,00000033,5844,58.863792,121.662415,159.491696,3.0,True
1,train,00000002,7243,11.983985,5.751693,566.889196,160.0,True
53,train,00000066,4325,8.994220,6.561684,643.200632,1.0,True
35,train,00000043,2322,7.149009,11.049285,193.974866,230.0,True
64,train,00000081,3224,4.435484,4.530170,166.249060,536.0,False
76,train,00000096,3271,4.432895,4.439645,109.220986,1.0,False
67,train,00000084,16031,3.692845,6.548253,103.958519,312.0,False
14,train,00000019,2354,1.656754,5.515967,295.816956,1.0,False
15,train,00000020,2353,1.529962,5.424233,158.586567,1.0,False


## Step 4: candidates and features for one trip

Candidates = every arc within R metres of the GPS point (spatial index), keeping the K nearest. Uses only GPS and the road network, never the route, so the API can reuse it.

In [8]:
from shapely import STRtree, linestrings, points as shp_points

def build_features(track, nodes, arcs, R=50.0, K=20):
    lon0, lat0 = track[0, 0], track[0, 1]
    gx, gy = to_xy(track[:, 0], track[:, 1], lon0, lat0)
    ts = track[:, 2]
    x, y = to_xy(nodes[:, 0], nodes[:, 1], lon0, lat0)
    ax, ay, bx, by = x[arcs[:, 0]], y[arcs[:, 0]], x[arcs[:, 1]], y[arcs[:, 1]]
    dx, dy = bx - ax, by - ay
    L2 = dx * dx + dy * dy
    L = np.sqrt(L2)
    out_deg = np.bincount(arcs[:, 0], minlength=len(nodes))
    in_deg = np.bincount(arcs[:, 1], minlength=len(nodes))

    vx, vy = np.gradient(gx, ts), np.gradient(gy, ts)
    speed = np.hypot(vx, vy)
    d_t = np.diff(ts)
    dt_prev, dt_next = np.r_[0.0, d_t], np.r_[d_t, 0.0]
    heading_ok = (speed > 1.0) & (np.maximum(dt_prev, dt_next) <= 5)

    lines = linestrings(np.stack([np.stack([ax, ay], 1), np.stack([bx, by], 1)], 1))
    pt, arc = STRtree(lines).query(shp_points(gx, gy), predicate='dwithin', distance=R)

    t = np.clip(((gx[pt] - ax[arc]) * dx[arc] + (gy[pt] - ay[arc]) * dy[arc])
                / np.where(L2[arc] == 0, 1, L2[arc]), 0, 1)
    cx, cy = ax[arc] + t * dx[arc], ay[arc] + t * dy[arc]
    dist = np.hypot(gx[pt] - cx, gy[pt] - cy)
    side = np.sign(dx[arc] * (gy[pt] - ay[arc]) - dy[arc] * (gx[pt] - ax[arc]))
    cos_h = (vx[pt] * dx[arc] + vy[pt] * dy[arc]) / (speed[pt] * L[arc] + 1e-9)
    cos_h = np.where(heading_ok[pt] & (L[arc] > 0), cos_h, 0.0)

    df = pd.DataFrame({
        'pt': pt, 'arc': arc, 'dist': dist, 't_frac': t, 'along_m': t * L[arc], 'arc_len': L[arc],
        'lateral': side * dist, 'heading_cos': cos_h, 'heading_ok': heading_ok[pt].astype(np.int8),
        'speed': speed[pt], 'dt_prev': dt_prev[pt], 'dt_next': dt_next[pt],
        'out_deg_to': out_deg[arcs[arc, 1]], 'in_deg_from': in_deg[arcs[arc, 0]]})
    df['n_cand'] = df.groupby('pt')['pt'].transform('size')
    df = df.sort_values(['pt', 'dist'], kind='stable')
    df['rank'] = df.groupby('pt').cumcount()
    df = df[df['rank'] < K].copy()
    df['d_gap'] = df['dist'] - df.groupby('pt')['dist'].transform('min')
    return df.reset_index(drop=True)

f = build_features(track, nodes, arcs)
f['y'] = (f['arc'].values == label_arc[f['pt'].values]).astype(int)
print(f.shape, '| candidates per point: mean %.1f, max %d' % (f.groupby('pt').size().mean(), f.groupby('pt').size().max()))
f.head()

(13086, 18) | candidates per point: mean 5.2, max 20


,pt,arc,dist,t_frac,along_m,arc_len,lateral,heading_cos,heading_ok,speed,dt_prev,dt_next,out_deg_to,in_deg_from,n_cand,rank,d_gap,y
0,0,17630,6.073087,0.872784,312.154332,357.653620,-6.073087,0.970293,1,23.591233,0.0,1.0,3,3,6,0,0.000000e+00,1
1,0,17634,6.073087,0.127216,45.499288,357.653620,6.073087,-0.970293,1,23.591233,0.0,1.0,3,3,6,1,3.552714e-15,0
2,0,17668,27.534465,0.859638,224.935686,261.663344,27.534465,0.861978,1,23.591233,0.0,1.0,3,3,6,2,2.146138e+01,0
3,0,17635,27.534465,0.140362,36.727658,261.663344,-27.534465,-0.861978,1,23.591233,0.0,1.0,3,3,6,3,2.146138e+01,0
4,0,212,45.902806,1.000000,33.122182,33.122182,45.902806,-0.969814,1,23.591233,0.0,1.0,3,2,6,4,3.982972e+01,0


## Step 5: build the dataset for all 90 trips and save it

In [9]:
R, K = 50.0, 20
frames, pts = [], []
t0 = time.time()
for i, ((part, tid), (lab, _)) in enumerate(LABELS.items()):
    trk, nds, arc, _ = load_trip(part, tid)
    f = build_features(trk, nds, arc, R=R, K=K)
    f['y'] = (f['arc'].values == lab[f['pt'].values]).astype(np.int8)   # 1 = this candidate is the true arc
    f.insert(0, 'trip', tid)
    f.insert(0, 'part', part)
    frames.append(f)
    pts.append(dict(part=part, trip=tid, n_points=len(trk)))
    if i % 10 == 0:
        print(i, part, tid, 'rows so far:', sum(len(x) for x in frames), '%.0f s' % (time.time() - t0))

df = pd.concat(frames, ignore_index=True)
for c in df.select_dtypes('float64').columns:
    df[c] = df[c].astype('float32')

os.makedirs(os.path.join('data', 'processed'), exist_ok=True)
df.to_csv(os.path.join('data', 'processed', 'candidates.csv.gz'), index=False)
pd.DataFrame(pts).to_csv(os.path.join('data', 'processed', 'trip_points.csv'), index=False)
print('saved', df.shape)

0 train 00000001 rows so far: 14096 0 s


10 train 00000013 rows so far: 253845 1 s


20 train 00000026 rows so far: 396812 2 s


30 train 00000037 rows so far: 617022 3 s


40 train 00000048 rows so far: 777152 4 s


50 train 00000062 rows so far: 960046 5 s


60 train 00000075 rows so far: 1122487 6 s


70 train 00000088 rows so far: 1387278 7 s


80 student_test 00000000 rows so far: 1643463 8 s


saved (1814494, 20)


## Step 6: sanity checks

In [10]:
P = pd.DataFrame(pts)
n_points = P.groupby('part')['n_points'].sum()
hit = df.groupby(['part', 'trip', 'pt'])['y'].max()
nearest_ok = df[df['rank'] == 0].groupby('part')['y'].sum()

print('points per partition:', n_points.to_dict())
print('candidate miss rate (%):', (100 * (1 - hit.groupby('part').sum() / n_points)).round(2).to_dict())
print('nearest-candidate top-1 accuracy (%):', (100 * nearest_ok / n_points).round(2).to_dict())
print('positive rate among candidate rows (%):', (100 * df.groupby('part')['y'].mean()).round(2).to_dict())
print('mean candidates per point:', round(df.groupby(['part', 'trip', 'pt']).size().mean(), 1))
print('NaNs in features:', int(df.drop(columns=['part', 'trip']).isna().sum().sum()))
print('trips per partition:', df.groupby('part')['trip'].nunique().to_dict())
print('trips shared between train and student_test:', set(df[df.part == 'train'].trip) & set(df[df.part == 'student_test'].trip))

# same numbers excluding the suspect-label trips
ok_trips = set(Q.loc[~Q['suspect'], 'trip'])
hit_ok = hit[hit.index.get_level_values('trip').isin(ok_trips)]
n_ok = P[P.trip.isin(ok_trips)].groupby('part')['n_points'].sum()
print('\nmiss rate (%) excluding suspect trips:', (100 * (1 - hit_ok.groupby('part').sum() / n_ok)).round(2).to_dict())

points per partition: {'student_test': 19160, 'train': 208417}
candidate miss rate (%): {'student_test': 0.15, 'train': 11.42}
nearest-candidate top-1 accuracy (%): {'student_test': 52.43, 'train': 49.06}


positive rate among candidate rows (%): {'student_test': 10.39, 'train': 11.32}


mean candidates per point: 8.7
NaNs in features: 0


trips per partition: {'student_test': 10, 'train': 80}


trips shared between train and student_test: set()

miss rate (%) excluding suspect trips: {'student_test': 0.15, 'train': 0.72}
